# C.3 多試幾次能找到正確答案嗎？


假設固定一道題每次答對率p=0.3，從同一分佈獨立抽兩份。兩次都錯的機率0.7²=0.49，所以至少一份對是0.51。多抽能增加候選中含正解的機會，還沒有說明最後怎樣選。

一般為 `1-(1-p)**n`：先算n份全失敗，再從1扣。候選集合命中稱coverage；pass@k也描述k份至少一份通過，但要依具體採樣與估計方法說明。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
p = 0.3
for n in (1, 2, 4, 8):
    coverage = 1 - (1 - p) ** n
    print("候選數", n, "獨立假設下包含正解", round(coverage, 6))
candidates = [3, 4, 3, 5]
print("本組候選", candidates, "包含真值4", 4 in candidates)

候選數 1 獨立假設下包含正解 0.3
候選數 2 獨立假設下包含正解 0.51
候選數 4 獨立假設下包含正解 0.7599
候選數 8 獨立假設下包含正解 0.942352
本組候選 [3, 4, 3, 5] 包含真值4 True


四個人工假設結果0.3、0.51、0.7599、0.942352。固定清單含4也只是人寫的例子，沒有生成或量出p。把候選改成全3，只改變清單命中，不會自動把公式中的p重新估計。

獨立假設針對固定題、同一分佈與互不影響亂數。重複相同錯解不必然違反獨立，它可能只是該題成功率很低。不同題的p也不同，不能把平均單次正確率直接代公式預測所有題。

確定生成每次同輸入同輸出，若固定錯誤，多調用沒有新機會；若提示依前一答案改寫，就另外檢查分佈與獨立前提。實際保存多題候選，逐份查真值，再算集合命中。即使集合有4，最後選第一份3還是失敗，下一節固定同一集合比較選法。

<details>
<summary>補充：舊候選生成與pass@k的計數關係</summary>

正式組沿用[C.1的兩種回答訓練](https://birdhackor.github.io/tiny-perceptron-vlm/C.1.html)：兩支模型從同一份隨機初值分開訓練，短答學最後整數，步驟支線學可驗算的中間式。例如同一道`(1+2)+3=?`，短答示範是`6`，步驟示範是`1+2=3;3+3=6;answer=6`；這是訓練目標的例子，實際生成仍可能錯。兩支更新次數相同，但步驟支線讀過更多回答token，並非等文字預算比較。

兩支對同一24道留出題，在k=1、2、4、8各自重新抽一批候選；溫度固定0.7。採樣期間各支保持自己的訓練後權重，不是兩欄共用同一份訓練後權重。每格計「本次集合中至少一份最後答案正確」的題數，分母都24：

| 每題候選k | 短答oracle coverage | 步驟oracle coverage |
| --- | --- | --- |
| 1 | 1/24 | 11/24 |
| 2 | 1/24 | 13/24 |
| 4 | 2/24 | 13/24 |
| 8 | 1/24 | 14/24 |

本表將已知算術真值下的實際集合命中比例標為oracle coverage，沒有用假設的p代替生成。[標準pass@k估計式](https://arxiv.org/pdf/2107.03374v2)見原論文§2.1式(1)：從每題n份候選、其中c份成功，估計k份裡至少一份成功的機會。當只抽k份，也就是n=k，這個式子恰好化成「本組是否有成功」：c=0時為0，c大於0時為1。因此本表的平均命中比例與這個特例一致；本輪沒有從較大的n候選池估計各k，也沒有執行HumanEval benchmark。

不同k的集合是重新採樣，沒有逐份追加或互相包含的保證，所以短答從2/24回到1/24並不矛盾。單次有限採樣不保證曲線單調；每支共保留360份候選與原始ID，仍只測24道不同題目。命中比例也只問集合含不含正解，不能替代能部署的選擇器成績。

</details>
